# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [8]:
!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

In [22]:
import os
import getpass

# First set up LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")

# Then set up OpenAI separately
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

# Now initialize the models with the correct API key
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

We'll also want to set a project name to make things easier for ourselves.

In [19]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [20]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [4]:
!mkdir data

mkdir: data: File exists


In [5]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31440    0 31440    0     0   216k      0 --:--:-- --:--:-- --:--:--     0--:-- --:--:--  216k


In [6]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70299    0 70299    0     0   687k      0 --:--:-- --:--:-- --:--:--  693k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

This code scans the data/ folder, finds all .html files, loads them, and stores them in the docs variable.

In [9]:
from langchain_community.document_loaders import DirectoryLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


In [25]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,  # Smaller chunk size to avoid token limits
    chunk_overlap=200
)

# Split the documents
smaller_docs = []
for doc in docs:
    splits = text_splitter.split_text(doc.page_content)
    for i, split in enumerate(splits):
        smaller_docs.append(Document(
            page_content=split,
            metadata={**doc.metadata, "chunk": i}
        ))

# Use smaller_docs instead of docs for creating the knowledge graph
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

kg = KnowledgeGraph()
for doc in smaller_docs:  # Note: using smaller_docs here
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

This code initializes an LLM and an embedding model for use in the RAG pipeline. It wraps OpenAI's gpt-4o model for text generation and OpenAIEmbeddings for vector embeddings.

In [26]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

This code initializes an empty KnowledgeGraph object from the ragas.testset.graph module and assigns it to the variable kg. The last line outputs the created knowledge graph.

This code merges multiple tqdm variants (autonotebook, asyncio, std) into a single interface, suppressing warnings and selecting the best implementation. The trange function is a shortcut for tqdm(range(...)).

In [27]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

This code adds each document from docs as a Node of type DOCUMENT to the KnowledgeGraph (kg), storing its content and metadata.

In [28]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

This code applies default transformations (e.g., summarization, theme extraction) to the KnowledgeGraph (kg) using the generator_llm and generator_embeddings, modifying the graph based on the document content.

In [29]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

KnowledgeGraph(nodes: 14, relationships: 71)

We can save and load our knowledge graphs as follows.

In [30]:
kg.save("ai_across_years_kg.json")
ai_across_years_kg = KnowledgeGraph.load("ai_across_years_kg.json")
ai_across_years_kg

KnowledgeGraph(nodes: 14, relationships: 71)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [31]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=ai_across_years_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

This code defines a query distribution for synthetic test data generation, using three types of query synthesizers with specified probabilities.

In [32]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

LSY ANSWER -><br>
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
            creates simple questions that need just one piece of information<br> 
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
            creates questions that need you to connect data between different parts of a document<br>
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
            creates questions that need multiple specific facts, like asking to combine details from different sections of a document<br>

Finally, we can use our `TestSetGenerator` to generate our testset!

In [33]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Samples: 100%|██████████| 11/11 [00:11<00:00,  1.07s/it]


,user_input,reference_contexts,reference,synthesizer_name
0,Wht r Large Langauge Models?,[Code may be the best application The ethics o...,Large Language Models are a type of software c...,single_hop_specifc_query_synthesizer
1,What be the connection between AGI and gullibi...,[Based Development As a computer scientist and...,The text suggests that solving the problem of ...,single_hop_specifc_query_synthesizer
2,What Simon say about weblog?,[Simon Willison’s Weblog Subscribe Stuff we fi...,Simon Willison’s Weblog includes a post titled...,single_hop_specifc_query_synthesizer
3,Wht did Google say in the leaked document?,[easy to follow. The rest of the document incl...,"The leaked Google document stated, ""We Have No...",single_hop_specifc_query_synthesizer
4,"What OpenAI doin' with GPT-4, why it ain't top...",[Prompt driven app generation is a commodity a...,"OpenAI's GPT-4, which was their best model alm...",single_hop_specifc_query_synthesizer
5,How do the challenges of LLMs as black boxes r...,[<1-hop>\n\nCode may be the best application T...,The challenges of LLMs as black boxes are intr...,multi_hop_abstract_query_synthesizer
6,How do the challenges of LLMs being black boxe...,[<1-hop>\n\nCode may be the best application T...,The challenges of LLMs being black boxes and t...,multi_hop_abstract_query_synthesizer
7,What are the ethical implications of using Lar...,[<1-hop>\n\nCode may be the best application T...,The ethical implications of using Large Langua...,multi_hop_abstract_query_synthesizer
8,How have the advancements in GPT-4o and the re...,[<1-hop>\n\ngets you OpenAI’s most expensive m...,"The advancements in GPT-4o, along with the red...",multi_hop_specific_query_synthesizer
9,How has the development of Claude 3 and its pr...,[<1-hop>\n\nThose of us who understand this st...,"The development of Claude 3, particularly its ...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



Shortcut which bypasses the manual setup of a KnowledgeGraph and query distribution. Above, we created a KG, added documents as nodes, applied transformations, and defined query synthesizers, then generated test queries manually. below method automates those steps with calling generate_with_langchain_docs(docs, testset_size=10).

**manual approach:**
If you need custom query distributions (e.g., more complex multi-hop queries).
If your dataset requires specific transformations (e.g., entity linking, summarization).
If you want greater reproducibility and insight into how queries are generated.<br>
<br>
**shortcut, straight call to generate_with_langchain_docs(docs, testset_size=10).**
If default settings are good enough for your use case.
If you don’t need fine-tuned control over query generation.
If you just need quick test data without customization.

In [35]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Generating Samples: 100%|██████████| 12/12 [00:36<00:00,  3.04s/it]


In [40]:
dataset.to_pandas()


,user_input,reference_contexts,reference,synthesizer_name
0,What role did OpenAI play in the development o...,[Code may be the best application The ethics o...,"A year ago, the only organization that had rel...",single_hop_specifc_query_synthesizer
1,What is the role of the ChatGPT Code Interpret...,[Based Development As a computer scientist and...,The ChatGPT Code Interpreter allows the LLM to...,single_hop_specifc_query_synthesizer
2,What significant developments in Large Languag...,[Simon Willison’s Weblog Subscribe Stuff we fi...,"In 2023, Large Language Models (LLMs) were rec...",single_hop_specifc_query_synthesizer
3,What is the significance of Stable Diffusion i...,[easy to follow. The rest of the document incl...,The context mentions that large language model...,single_hop_specifc_query_synthesizer
4,What are the ethical concerns related to the t...,[<1-hop>\n\nCode may be the best application T...,The ethical concerns related to the training d...,multi_hop_abstract_query_synthesizer
5,What are the ethical concerns related to the t...,[<1-hop>\n\nCode may be the best application T...,The ethical concerns related to the training d...,multi_hop_abstract_query_synthesizer
6,How do the training costs of large language mo...,[<1-hop>\n\nCode may be the best application T...,"The training costs of large language models, w...",multi_hop_abstract_query_synthesizer
7,How do the ethics of training data impact the ...,[<1-hop>\n\nCode may be the best application T...,The ethics of training data significantly impa...,multi_hop_abstract_query_synthesizer
8,What are the environmental and ethical implica...,[<1-hop>\n\nthat. DeepSeek v3 is a huge 685B p...,The training of large language models like Dee...,multi_hop_specific_query_synthesizer
9,How have advancements in GPT-4 and GPT-4o mode...,[<1-hop>\n\nPrompt driven app generation is a ...,Advancements in GPT-4 and GPT-4o models have s...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

This code initializes a LangSmith Client, checks if a dataset named "State of AI Across the Years" already exists, deletes it if found, and then creates a new dataset with the same name. This ensures that the dataset starts fresh without duplicates.

In [41]:
from langsmith import Client
from datetime import datetime

client = Client()

dataset_name = "State of AI Across the Years"

# Try to delete the existing dataset if it exists
try:
    existing_dataset = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_name=dataset_name)
except:
    pass  # Dataset doesn't exist, which is fine

# Create new dataset
langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="State of AI Across the Years!"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

loops through the dataset, converts it to a Pandas dataframe, and uploads each row as an example to LangSmith.
<br>        
inputs → The test question (user_input)<br>
outputs → The reference answer (reference)<br>
metadata → The context used for generating the answer (reference_contexts)<br>
Each example is linked to the previously created LangSmith dataset (langsmith_dataset.id).

In [42]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [43]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [44]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [45]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [46]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="State of AI"
)

converts the vectorstore into a retriever, allowing it to fetch relevant documents.

as_retriever() turns the vector database into a retrievable search tool.
search_kwargs={"k": 10} means it will return the top 10 most relevant documents for each query.

In [47]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

defines a prompt template for the RAG (Retrieval-Augmented Generation) model.

In [48]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [64]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

Finally, we can set-up our RAG LCEL chain!

This rag_chain:

Takes a question,
Retrieves relevant documents,
Formats the input with a prompt,
Generates an answer using an LLM,
Returns the final response as text.

In [66]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [67]:
rag_chain.invoke({"question" : "What are Agents?"})

'Agents are described as AI systems that can act on your behalf, though the term is considered vague and lacks a clear definition. Some people envision agents as similar to travel agents, while others think of them as LLMs (Large Language Models) that have access to tools to solve problems. The discussion around agents often includes the concept of autonomy, but overall, there is skepticism about their utility due to issues like gullibility.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4o as our evaluation LLM for our base Evaluators.

initializes the evaluation LLM as gpt-4o. separates the evaluation LLM from the generator LLM.

In [68]:
eval_llm = ChatOpenAI(model="gpt-4o")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

 code defines three evaluation metrics using LangChainStringEvaluator, each using eval_llm (gpt-4o) to assess responses.

In [70]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dope_or_nope_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this submission dope, lit, or cool?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

LSY ANSWER -> 
- `qa_evaluator`: Evaluates the accuracy of the answer by comparing it to the reference answer.
- `labeled_helpfulness_evaluator`: Evaluates the helpfulness of the answer by considering the relevance to the user's question and the correctness of the reference answer.
- `dope_or_nope_evaluator`: Evaluates the coolness of the answer by checking if it's dope, lit, or cool.

## LangSmith Evaluation

This runs an automated evaluation of rag_chain responses, comparing them against reference answers and scoring them based on accuracy, helpfulness, and style.

In [50]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'back-meat-58' at:
https://smith.langchain.com/o/fb0d5d7c-8599-4482-8163-c1156c9773c9/datasets/6a46751c-70f8-4da7-8020-b78327aba3d4/compare?selectedSessions=196902d8-d505-4cb1-946f-592822af61e2




12it [02:24, 12.00s/it]


,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How has the development of Claude 3 and its su...,I don't know.,None,The development of Claude 3 and its subsequent...,0,0,0,3.667224,9e452c39-e383-40df-8939-0ad43669c95a,875be98b-4811-4b1c-9c91-e96914a01f24
1,What advancements in large language models wer...,"In 2024, significant advancements in large lan...",None,"In 2024, significant advancements in large lan...",1,0,1,2.734534,62325937-d2fa-4fe3-b0fc-639ca8476bff,91cde943-8c53-479b-b10f-84bc6b46f5bc
2,How has Google's Gemini 1.5 Flash model contri...,I don't know.,None,Google's Gemini 1.5 Flash model has significan...,0,0,0,1.041966,12d472f9-8fbd-4032-9949-0da0c10523ec,484d67be-9dab-4421-b454-b316e6bc3507
3,What happen with large language models in 2023...,"In 2023, Large Language Models (LLMs) experien...",None,"In 2023, large language models (LLMs) were rec...",1,1,1,3.774387,876fe1a2-97de-43ca-8c25-cdb669d15fbb,ff89eb4b-7305-4b63-bff3-1d6b60e43589
4,How have advancements in large language models...,"Advancements in large language models, such as...",None,"Advancements in large language models, such as...",1,0,0,3.927562,cfb7fba3-3f47-4bca-983b-fc8ba07343ca,0817e456-05e9-4d4a-9da3-0932eae5ee92
5,How have inference-scaling reasoning models an...,"Inference-scaling reasoning models, such as Op...",None,Inference-scaling reasoning models have been a...,1,0,0,2.838548,71a343ef-936b-4ad6-addf-be5bdae84fe4,287d6a61-526c-4d09-98f7-873d401f88cb
6,How have inference-scaling reasoning models an...,"Inference-scaling reasoning models, exemplifie...",None,Inference-scaling reasoning models have been a...,1,0,1,7.529293,541fd9c7-9998-42bc-bd4a-fb4f5e96059a,662c347b-2071-4518-8d50-c0dd8ae3a7ba
7,How have inference-scaling reasoning models an...,I don't know.,None,"In 2024, the advancements in large language mo...",0,0,0,0.869657,9cecc220-6ddf-45d8-8af9-ef3e106ba88e,a4322375-d37f-4edc-9787-be3c537ee1d4
8,Wht are the Plausible analytics insights from ...,I don't know.,None,The AI Research Analyst used Plausible analyti...,0,0,0,0.875736,ad4d804f-9984-4feb-b4cd-7542f6cb7380,bada1688-e1d2-490d-9ad4-09ba22064acc
9,Wht are the key insights from Simon Willison's...,Key insights from Simon Willison's Weblog abou...,None,Simon Willison's Weblog highlights that 2023 w...,1,0,0,2.595741,b148588d-9c6d-4726-9136-5c26eef82d50,38786e8d-674b-4a52-96e1-254a18d8f1c3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

This prompt ensures retrieval-based responses but adds personality, making them sound more engaging and dope.

In [ ]:
DOPE_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the questions in a dope way, be cool!

Context: {context}
Question: {question}
"""

dope_rag_prompt = ChatPromptTemplate.from_template(DOPE_RAG_PROMPT)

In [52]:
rag_documents = docs

In [53]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

LSY ANSWER -><br> 
Retrieval Efficiency – Smaller chunks improve granularity but increase retrieval latency; larger chunks reduce retrieval calls but may dilute relevance.<br>
Embedding & Indexing Cost – Smaller chunks create more embeddings, increasing storage and processing overhead.<br>
Response Quality – Smaller chunks risk losing context; larger chunks retain context but may retrieve irrelevant content.<br>
Overlap & Redundancy – Higher chunk_overlap improves continuity but increases redundancy.<br>
Latency & Performance – Smaller chunks mean more retrievals, increasing processing time; larger chunks improve speed but risk lower relevance.

In [72]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?
LSY ANSWER -><br>    
Embedding Quality – Larger models (e.g., text-embedding-3-large) capture richer semantics, improving retrieval accuracy, while smaller models may miss nuances.<br>
Indexing & Storage Cost – Higher-dimensional embeddings require more storage and increase indexing time.<br>
Retrieval Latency – Larger embeddings slow down vector search and increase compute cost, while smaller embeddings speed up retrieval.<br>
Model Cost – More advanced models incur higher API costs per call.
Downstream Task Performance – Better embeddings improve response relevance but may introduce diminishing returns if retrieval isn't the bottleneck.

This code creates an in-memory vector database using Qdrant to store and retrieve document embeddings for the RAG pipeline.

In [73]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Across Years (Augmented)"
)

This code converts the Qdrant vector database into a retriever for use in the RAG pipeline.

In [74]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [75]:
dope_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dope_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

This code defines the Dope-RAG pipeline, which retrieves documents, formats a "cool" prompt, and generates an answer.

In [76]:
dope_rag_chain.invoke({"question" : "what are Agents?"})

"Agents? Man, they're like this elusive concept in the AI world—everyone's talking about them, but nobody's on the same page. Some folks think of agents as those trusty travel assistants that book your trips, while others see them as LLMs (that's large language models for the uninitiated) that can run tools to solve problems. But here's the kicker: the definition is still up for grabs, and it feels like they’re always “coming soon” but never really show up. It's a whole vibe of confusion, you know?"

Finally, we can evaluate the new chain on the same test set!

In [77]:
evaluate(
    dope_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "dope_chain"},
)

View the evaluation results for experiment: 'advanced-swim-34' at:
https://smith.langchain.com/o/fb0d5d7c-8599-4482-8163-c1156c9773c9/datasets/8e1db638-f405-42f8-bdb7-570ec6b44d02/compare?selectedSessions=cd3de519-5689-4895-833e-a166dd4b15c2




12it [02:30, 12.50s/it]


,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How did the advancements in GPT-4 and related ...,"Yo, the advancements in GPT-4 and its pals lik...",None,"In 2024, advancements in GPT-4 and related mod...",1,0,1,3.374231,55d04d41-1720-4844-822b-68b85133c5d0,c14f1601-3034-40db-b8b8-7c7a80b6181c
1,How have the developments in GPT-4o and its pr...,"Yo, check it out! The drop in pricing for GPT-...",None,"The developments in GPT-4o, particularly its p...",1,0,1,3.838885,d067b96c-47b6-444e-92a8-975ed4a3b637,fb0838b6-f98d-482a-951d-3902e6fe0d5d
2,How have advancements in GPT-4 and GPT-4o mode...,"Yo, let’s break it down! The advancements in G...",None,Advancements in GPT-4 and GPT-4o models have s...,1,1,1,3.175738,453ce207-4fda-48e0-a14c-5b6e4be5807d,8bbda09a-3153-4ece-a48c-a3177a8fee2f
3,What are the environmental and ethical implica...,I don't know.,None,The training of large language models like Dee...,0,0,0,0.861865,3d766302-b66b-422b-a90e-775f9f79ec07,e39a12ef-0162-40b4-8268-9cfdf0e04ec5
4,How do the ethics of training data impact the ...,"Yo, the ethics of training data are a total ga...",None,The ethics of training data significantly impa...,1,1,1,3.378321,45f2f179-0f56-4b81-a28e-0f56931a100e,2457df0c-a1f7-4ca9-8805-e39ab89eff55
5,How do the training costs of large language mo...,"Yo, that's a deep question! So, while the cont...",None,"The training costs of large language models, w...",1,0,1,2.512718,77364f04-819e-4095-afee-ad3607461cd5,a3470821-eeb3-4cea-9881-54887b4945e1
6,What are the ethical concerns related to the t...,"Yo, the ethical concerns about training data f...",None,The ethical concerns related to the training d...,1,0,1,2.541200,2d333be6-2ad0-43d3-b5f9-6580d0a41a3c,56a5c7bd-f50d-434b-85af-82acd9e7d2df
7,What are the ethical concerns related to the t...,"Yo, the ethical concerns around training data ...",None,The ethical concerns related to the training d...,1,0,1,3.927331,e6186329-7c1b-4c8f-ada5-0f0bc0328ad0,d688166e-da20-4cde-a7b1-a0fe27d7c037
8,What is the significance of Stable Diffusion i...,Stable Diffusion marked a pivotal moment for l...,None,The context mentions that large language model...,1,1,1,8.686878,06177fa3-7fc7-4ffc-a175-32c6da231445,a28e9317-3ba7-4987-9aba-94ac243eebef
9,What significant developments in Large Languag...,"In 2023, we saw some epic moments for Large La...",None,"In 2023, Large Language Models (LLMs) were rec...",1,0,1,2.349129,b109befe-c16a-471c-b228-866a4ef28e02,a60899f7-ccca-4374-bb0c-d253a5fa1c7f


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

first is the generic rag chain:

![dope](./images/not_dope.png)


next is the dope rag chain:

![dope](./images/dope1.png)

differences between the two chains:<br>         
prompt: rag_chain is the generic prompt; dope_rag_chain is the dope prompt with more fluent, cool responses<br>
chunk size: rag_chain is 500; dope_rag_chain is 1000<br>
embedding model: rag_chain is text-embedding-3-small; dope_rag_chain is text-embedding-3-large<br>

dope_rag_chain improves fluency, retrieval accuracy, and coherence through better prompting, embeddings, and chunking. It might have slightly higher latency due to larger embeddings and chunk size, but it's more accurate and fluent.

